# Arrays and Plotting

## Vectorization of Stepwise Functions

The Heaviside function is defined as:
$$
H(x) = \begin{cases}
    0 & x < 0 \\
    1 & x \ge 0
\end{cases}
$$
To implement this in Python, the simplest way looks something like this:

In [ ]:
def Heaviside(x):
    """Heavidside function, defined as:
        0, x < 0
        1, x >= 0
    """
    if x < 0:
        return 0
    return 1

However, this particular function doesn't work on arrays beacuse the `if` statement expects a single conditional statement, while `x < 0` returns an array of booleans:

In [1]:
import numpy as np

x = np.array([-2, -1, 0, 1, 2])
print(x < 0)

[ True  True False False False]


There are several ways to improve this function so that it works properly when passed an array argument.

### Manual Loop

The first method is manually loop over values in an array:

In [ ]:
def Heaviside_loop(x):
    """Array-safe implementation of Heaviside function with loops"""
    x = np.asarray(x)           # Turn into array so that subsequent code works
    r = np.zeros(len(x))        # Make output array of same size as input
    for i in range(len(x)):
        r[i] = Heaviside(x[i])  # Pass scalar to existing Heaviside function
    return r

While this is straightforward, it doesn't include vectorization (skipping the `for` loop) and is generally a slower solution. There are are many ways to address this, but we'll skip ahead to a general vectorized approach.

### Manual Vectorization

The general outline using NumPy methods takes functions of the form:
```python
def f(x):
    if condition1:
        r = <expression 1>
    elif condition2:
        r = <expression 2>
    return r
```
and puts them in the vectorized form:
```python
def f(x):
    x = np.asarray(x)

    c1 = condition1
    c2 = condition2

    r = np.zeros_like(x)
    r[c1] = <expression 1>
    r[c2] = <expression 2>
    return r

```
The code above uses *boolean indexing*, which only evaluates the assignment operator (`=`) if the condition is `True`. We'll talk more about how this works in the below examples. It also uses the function `np.zeros_like()` to make a copy of `x`. This avoids calling `len(x)` which doesn't work if we pass a scalar!

This method can be used with as many conditional statements as you need. Note that it's written with any potential `else` statement at the end translated into an `elif` statement that can be passed to the `np.where()` function.

In the case of the Heaviside function, we could write:

In [2]:
def Heaviside_vector(x):
    """Array-safe implementation of Heaviside function with vectorization"""
    x = np.asarray(x)

    c1 = x < 0
    c2 = x >= 0

    result = np.zeros_like(x) # create output with same size as input
    result[c1] = 0
    result[c2] = 1

    return result

print(Heaviside_vector(1.0))
print(Heaviside_vector([-1.0, 0.0, 1.0]))

1.0
[0. 1. 1.]


### Chained Conditional Statements

When using the method above, you need to be careful about using conditional chains. For example, consider the hat function:
$$
h(x) = \begin{cases}
    0 & x < 0 \\
    x & 0 \le x < 1 \\
    2-x & 1 \le x < 2 \\
    0 & x \ge 2
\end{cases}
$$
If you pass your Python function an array and use chained comparisons, it will get confused:

In [6]:
x = np.array([-1, 0, 1])
print(0 <= x)
print(x < 1)
print(0 <= x & x < 1) # uncomment and try me after running the above two lines

[False  True  True]
[ True  True False]


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

This is because Python evaluates the first conditions `0 <= x` first, which results in `[False True True]`. The second condition `x < 1` results in `[True True False]`. The chained comparison tries to take the `and` of these, but doesn't know how to handle two arrays.

To get around this, you can use the *logical* AND operator (`&`). This operator performs a piecwise comparison of each element in two lists, such that the result for the $i$-th element is the `and` of the $i$-th element from each input list.

We can now go ahead defining the array-safe hat function:

In [ ]:
def hat(x):
    """Hat function, with peak between 0 and 2"""
    x = np.asarray(x)

    # List of conditions
    c1 = x < 0
    c2 = (0 <= x) & (x < 1)
    c3 = (1 <= x) & (x < 2)
    c4 = x >= 2

    # Evaluate function for different conditions
    result = np.zeros_like(x)
    result[c1] = 0
    result[c2] = x[c2]
    result[c3] = 2 - x[c3]
    result[c4] = 0

    return result

print(hat([-1,0,1,2,3]))

## Practice

Define an array-safe function that returns an upside down hat:
$$
p(x) = \begin{cases}
1 & x < 0 \\
1-x & 0 \le x < 1 \\
x-1 & 1 \le x < 2 \\
1 & x \ge 2
\end{cases}